In [2]:
import pandas as pd
import os
from pathlib import Path

In [ ]:
# Auto-discovery of latest dataset
import glob

datasets_dir = "../data/phase0/"
pattern = os.path.join(datasets_dir, "mbpp_with_complexity_*.parquet")
matching_files = glob.glob(pattern)

if matching_files:
    file_path = max(matching_files, key=os.path.getmtime)
    print(f"Using: {Path(file_path).name}")
else:
    raise FileNotFoundError(f"No files found in {datasets_dir}")

In [ ]:
# Load and inspect
df = pd.read_parquet(file_path)
print(f"Records: {len(df):,}, Columns: {list(df.columns)}")
print(f"File size: {os.path.getsize(file_path) / (1024**2):.2f} MB")

In [ ]:
# Column info
df.info()

In [6]:
# Display first 10 records - full table view
# Set pandas display options to show full content
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)     # Show all rows (for head(10))
pd.set_option('display.max_colwidth', None) # Show full textin each cell
pd.set_option('display.width', None)        # Don't wrap to terminal width
print("First 10 records (complete table):")
df.head(3)

First 10 records (complete table):


,task_id,text,code,test_list,cyclomatic_complexity
0,1,"Write a function to find the minimum cost path to reach (m, n) from (0, 0) for the given cost matrix cost[][] and a position (m, n) in cost[][].","R = 3\r\nC = 3\r\ndef min_cost(cost, m, n): \r\n\ttc = [[0 for x in range(C)] for x in range(R)] \r\n\ttc[0][0] = cost[0][0] \r\n\tfor i in range(1, m+1): \r\n\t\ttc[i][0] = tc[i-1][0] + cost[i][0] \r\n\tfor j in range(1, n+1): \r\n\t\ttc[0][j] = tc[0][j-1] + cost[0][j] \r\n\tfor i in range(1, m+1): \r\n\t\tfor j in range(1, n+1): \r\n\t\t\ttc[i][j] = min(tc[i-1][j-1], tc[i-1][j], tc[i][j-1]) + cost[i][j] \r\n\treturn tc[m][n]","[assert min_cost([[1, 2, 3], [4, 8, 2], [1, 5, 3]], 2, 2) == 8, assert min_cost([[2, 3, 4], [5, 9, 3], [2, 6, 4]], 2, 2) == 12, assert min_cost([[3, 4, 5], [6, 10, 4], [3, 7, 5]], 2, 2) == 16]",7
1,2,Write a function to find the similar elements from the given two tuple lists.,"def similar_elements(test_tup1, test_tup2):\r\n res = tuple(set(test_tup1) & set(test_tup2))\r\n return (res)","[assert similar_elements((3, 4, 5, 6),(5, 7, 4, 10)) == (4, 5), assert similar_elements((1, 2, 3, 4),(5, 4, 3, 7)) == (3, 4), assert similar_elements((11, 12, 14, 13),(17, 15, 14, 13)) == (13, 14)]",1
2,3,Write a python function to identify non-prime numbers.,"import math\r\ndef is_not_prime(n):\r\n result = False\r\n for i in range(2,int(math.sqrt(n)) + 1):\r\n if n % i == 0:\r\n result = True\r\n return result","[assert is_not_prime(2) == False, assert is_not_prime(10) == True, assert is_not_prime(35) == True]",3


In [ ]:
# Complexity statistics
import numpy as np

complexity_scores = df['cyclomatic_complexity'].values
print(f"Complexity: min={complexity_scores.min()}, max={complexity_scores.max()}, mean={complexity_scores.mean():.2f}, median={np.median(complexity_scores):.1f}")
print(f"Percentiles: 25th={np.percentile(complexity_scores, 25):.1f}, 75th={np.percentile(complexity_scores, 75):.1f}, 90th={np.percentile(complexity_scores, 90):.1f}")

# Distribution
df['cyclomatic_complexity'].value_counts().sort_index()

In [ ]:
# Example record
example = df.iloc[0]
print(f"Problem {example['task_id']}: {example['text'][:80]}...")
print(f"Complexity: {example['cyclomatic_complexity']}")

In [ ]:
# Test prompt builder
import sys
sys.path.append('..')
from common.prompt_utils import PromptBuilder
import numpy as np

sample = df.iloc[0]
test_cases = sample.get('test_list', [])
if isinstance(test_cases, np.ndarray):
    test_cases = test_cases.tolist()
test_cases_str = '\n'.join(test_cases) if test_cases else "# No test cases"

prompt = PromptBuilder.build_prompt(
    problem_description=sample['text'],
    test_cases=test_cases_str
)
print(prompt)